In [1]:
## conda activate gift_spatial

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse

matplotlib.rcParams['pdf.fonttype'] = 42

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
from spatial_utils import read_dual_probe_data, read_gapfill_data, read_genotype_annotations, plot_wt_alt_alleles_spatial

import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


/data1/lareauc/users/blattms/miniconda3/envs/gift_spatial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
data_path = f"{large_data_dir}gf_CL_VisiumHD/"
# Load data:
gapfill_sdata = read_gapfill_data(gf_adata_path = f"{data_path}GapFill_CellLineMix_Visium_GF.h5ad", WTA_dir = f"{data_path}GapFill_WTA.zarr", cores = 4)
annotated_genotypes, celltype_genotypes, wt_alleles, alt_alleles = read_genotype_annotations()

/home/blattms1/projects/gapfill/gift_paper_reproducibility/1_figure_CL_proof_of_concept/code/spatial_utils.py:194: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  gapfill_wta = read_zarr(WTA_dir)
Genotyping : 100%|██████████| 159/159 [00:23<00:00,  6.77it/s, Probe ZNF609]                             
/data1/lareauc/users/blattms/miniconda3/envs/gift_spatial/lib/python3.12/site-packages/giftwrap/analysis/spatial.py:280: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  gf_table.uns[sd.models.TableModel.ATTRS_KEY] = wta_table.uns[sd.models.TableModel.ATTRS_KEY].copy()


Info: Calling genotypes for the binned data using the previous arguments:
flavor: basic
threshold: 0.5
cores: 4


Genotyping : 100%|██████████| 159/159 [00:08<00:00, 17.71it/s, Probe ZNF609]                             
/data1/lareauc/users/blattms/miniconda3/envs/gift_spatial/lib/python3.12/site-packages/giftwrap/analysis/spatial.py:280: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  gf_table.uns[sd.models.TableModel.ATTRS_KEY] = wta_table.uns[sd.models.TableModel.ATTRS_KEY].copy()


Info: Calling genotypes for the binned data using the previous arguments:
flavor: basic
threshold: 0.5
cores: 4


Genotyping : 100%|██████████| 159/159 [00:03<00:00, 40.75it/s, Probe ZNF609]                             
/data1/lareauc/users/blattms/miniconda3/envs/gift_spatial/lib/python3.12/site-packages/giftwrap/analysis/spatial.py:280: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  gf_table.uns[sd.models.TableModel.ATTRS_KEY] = wta_table.uns[sd.models.TableModel.ATTRS_KEY].copy()


In [4]:
resolutions = ['square_016um']

def mapping(x):
    x = x.split("|")
    if len(x) > 2:
        return " ".join(x[-2:])
    else:
        return " ".join(x)


for res in resolutions:
    print(f"Resolution: {res}")
    print("Original spatial library size:", gapfill_sdata.tables[res].X.sum())
    print("Gapfill spatial library size:", gapfill_sdata.tables[f'gf_{res}'].X.sum())

    wta_table = gapfill_sdata.tables[res]
    gf_table = gapfill_sdata.tables[f'gf_{res}']
    gf_table.var['probe'] = gf_table.var['probe'].str.split("|").str[-2:].str.join(" ")
    gf_table.var.index = gf_table.var.index.to_series().apply(mapping)

    gf_table.obs['cell_line'] = wta_table.obs['cell_line']

    # Pre-convert sparse matrices to dense arrays for fast indexing
    gf_X = gf_table.X.toarray() if issparse(gf_table.X) else np.asarray(gf_table.X)
    wta_X = wta_table.X.toarray() if issparse(wta_table.X) else np.asarray(wta_table.X)

    cell_lines = wta_table.obs['cell_line'].values
    barcodes = wta_table.obs_names.values
    wta_var_names = list(wta_table.var_names)

    # Precompute per-probe column indices and collect relevant WTA gene columns
    probe_names = gf_table.var['probe'].unique()
    probe_info = {}
    relevant_wta_cols = set()
    for probe_name in probe_names:
        probe_mask = (gf_table.var.probe == probe_name).values
        gene_name = gf_table.var.loc[probe_mask, 'gene'].values[0]
        wta_col = wta_var_names.index(gene_name) if gene_name in wta_var_names else None
        if wta_col is not None:
            relevant_wta_cols.add(wta_col)
        # Build per-cell-line correct/incorrect masks
        cl_masks = {}
        for cl in np.unique(cell_lines):
            expected_gf_list = celltype_genotypes[cl].get(probe_name, [])
            if not expected_gf_list:
                continue
            gf_selector = gf_table.var.gapfill.isin(expected_gf_list).values
            correct_cols = np.where(probe_mask & gf_selector)[0]
            incorrect_cols = np.where(probe_mask & ~gf_selector)[0]
            cl_masks[cl] = (correct_cols, incorrect_cols)
        if cl_masks:
            probe_info[probe_name] = (gene_name, wta_col, cl_masks)

    # Filter out cells with all zeros in both relevant WTA genes and all GF probes
    relevant_wta_cols = sorted(relevant_wta_cols)
    wta_relevant_sum = wta_X[:, relevant_wta_cols].sum(axis=1) if relevant_wta_cols else np.zeros(wta_X.shape[0])
    gf_sum = gf_X.sum(axis=1)
    nonzero_cells = (wta_relevant_sum > 0) | (gf_sum > 0)
    print(f"Cells before filtering: {len(nonzero_cells)}, after: {nonzero_cells.sum()}")

    # Vectorized accumulation
    rows = []
    for probe_name, (gene_name, wta_col, cl_masks) in probe_info.items():
        for cl, (correct_cols, incorrect_cols) in cl_masks.items():
            cl_idx = np.where((cell_lines == cl) & nonzero_cells)[0]
            if len(cl_idx) == 0:
                continue
            correct = gf_X[np.ix_(cl_idx, correct_cols)].sum(axis=1)
            incorrect = gf_X[np.ix_(cl_idx, incorrect_cols)].sum(axis=1)
            wta = wta_X[cl_idx, wta_col] if wta_col is not None else np.zeros(len(cl_idx))

            probe_df = pd.DataFrame({
                "Barcode": barcodes[cl_idx],
                "Cell Line": cl,
                "GF Probe": probe_name,
                "Gene": gene_name,
                "WTA Counts": wta.astype(int),
                "Correct Gapfill Counts": correct.astype(int),
                "Incorrect Gapfill Counts": incorrect.astype(int),
                "All Gapfill Counts": (correct + incorrect).astype(int),
            })
            rows.append(probe_df)

    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f'../output/visium_gapfill_counts_by_cell_line_unaggregated_{res}.csv.gz', index=False, compression='gzip')

Resolution: square_016um
Original spatial library size: 2.1254704e+08
Gapfill spatial library size: 859751.0
Cells before filtering: 61728, after: 60849
